# Fiche de Cours : Réseaux Récurrents Profonds (Deep RNN)

## Comparaison entre RNN Simple et Deep RNN

### Objectifs:
1. Comprendre l'architecture d'un RNN simple (1 couche)
2. Construire un Deep RNN (plusieurs couches empilées)
3. Comparer les performances et les représentations apprises

### Concepts clés:
- **Profondeur temporelle** : propagation de l'état caché à travers le temps
- **Profondeur verticale** : empilement de plusieurs couches récurrentes
- **Hiérarchie de représentation** : les couches basses captent des motifs locaux, les hautes captent des structures abstraites

## Cellule 1 : RNN Simple (1 couche) - Implémentation From Scratch

### Architecture et équations

Pour un RNN simple à un seul instant $t$:
$$H_t = \tanh(X_t W_{xh} + H_{t-1} W_{hh} + b_h)$$
$$Y_t = H_t W_{hq} + b_q$$

où:
- $X_t$ : entrée à l'instant $t$
- $H_t$ : état caché à l'instant $t$
- $W_{xh}$ : poids d'entrée → caché
- $W_{hh}$ : poids caché → caché (récurrence)
- $W_{hq}$ : poids caché → sortie

### Code PyTorch

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.optim import Adam


class SimpleRNNScratch(nn.Module):
    """RNN simple à 1 couche, implémentation from scratch."""
    
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        """
        Args:
            num_inputs: dimension de l'entrée
            num_hiddens: dimension de l'état caché
            sigma: écart-type de l'initialisation
        """
        super().__init__()
        self.num_inputs = num_inputs
        self.num_hiddens = num_hiddens
        
        
        self.W_xh = nn.Parameter(
            torch.randn(num_inputs, num_hiddens) * sigma
        )
        
        
        self.W_hh = nn.Parameter(
            torch.randn(num_hiddens, num_hiddens) * sigma
        )
        
        
        self.W_hq = nn.Parameter(
            torch.randn(num_hiddens, num_inputs) * sigma  # prédiction du prochain élément
        )
        
        
        self.b_h = nn.Parameter(torch.zeros(num_hiddens))
        self.b_q = nn.Parameter(torch.zeros(num_inputs))
    
    def forward(self, X, H=None):
        """
        Propagation avant.
        
        Args:
            X: (seq_len, batch_size, num_inputs) - séquence d'entrée
            H: (batch_size, num_hiddens) - état caché initial (None = zéro)
        
        Returns:
            outputs: liste des sorties à chaque instant
            H: état caché final
        """
        seq_len, batch_size, _ = X.shape
        
        
        if H is None:
            H = torch.zeros(batch_size, self.num_hiddens, device=X.device)
        
        outputs = []
        
        
        for t in range(seq_len):
            x_t = X[t]  
            
            H = torch.tanh(x_t @ self.W_xh + H @ self.W_hh + self.b_h)
            
            
            Y_t = H @ self.W_hq + self.b_q
            
            outputs.append(Y_t.unsqueeze(0))
        
        
        outputs = torch.cat(outputs, dim=0)  # (seq_len, batch_size, num_inputs)
        
        return outputs, H


print("✓ RNN Simple défini avec succès")
print(f"  - Architecture: 1 couche récurrente")
print(f"  - Équation: H_t = tanh(X_t @ W_xh + H_(t-1) @ W_hh + b_h)")

ModuleNotFoundError: No module named 'torch'

## Cellule 2 : Deep RNN (plusieurs couches) - Implémentation From Scratch

### Architecture mathématique

Pour un Deep RNN avec $L$ couches:

**Couche 1:**
$$H_t^{(1)} = \tanh(X_t W_{xh}^{(1)} + H_{t-1}^{(1)} W_{hh}^{(1)} + b_h^{(1)})$$

**Couches intermédiaires** $l \in \{2, ..., L\}$:
$$H_t^{(l)} = \tanh(H_t^{(l-1)} W_{xh}^{(l)} + H_{t-1}^{(l)} W_{hh}^{(l)} + b_h^{(l)})$$

**Sortie finale:**
$$Y_t = H_t^{(L)} W_{hq} + b_q$$

### Flux d'information
- La sortie de la couche $l$ à l'instant $t$ devient l'entrée de la couche $l+1$ au même instant
- Chaque couche maintient son propre état caché qui persiste dans le temps
- Les couches basses capturent des dépendances locales, les couches hautes des structures abstraites

### Code PyTorch

In [ ]:
class DeepRNNScratch(nn.Module):
    """Deep RNN avec plusieurs couches récurrentes empilées."""
    
    def __init__(self, num_inputs, num_hiddens, num_layers, sigma=0.01):
        """
        Args:
            num_inputs: dimension de l'entrée
            num_hiddens: dimension de l'état caché (même pour toutes les couches)
            num_layers: nombre de couches récurrentes
            sigma: écart-type de l'initialisation
        """
        super().__init__()
        self.num_inputs = num_inputs
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        
        # Créer num_layers couches RNN
        self.rnns = nn.ModuleList()
        for i in range(num_layers):
            # La première couche reçoit l'entrée originale
            input_size = num_inputs if i == 0 else num_hiddens
            
            # Créer une couche RNN simple
            rnn_layer = SimpleRNNScratch(input_size, num_hiddens, sigma)
            self.rnns.append(rnn_layer)
        
        # Couche de sortie finale (depuis la dernière couche cachée)
        self.W_hq = nn.Parameter(
            torch.randn(num_hiddens, num_inputs) * sigma
        )
        self.b_q = nn.Parameter(torch.zeros(num_inputs))
    
    def forward(self, X, Hs=None):
        """
        Propagation avant.
        
        Args:
            X: (seq_len, batch_size, num_inputs) - séquence d'entrée
            Hs: liste d'états cachés initiaux pour chaque couche
        
        Returns:
            outputs: (seq_len, batch_size, num_inputs) - sortie finale
            Hs: liste d'états cachés finaux pour chaque couche
        """
        seq_len, batch_size, _ = X.shape
        
        # Initialiser les états cachés de chaque couche
        if Hs is None:
            Hs = [None] * self.num_layers
        
        # Traiter la séquence à travers chaque couche
        outputs = X  # Entrée de la première couche
        
        for layer_idx in range(self.num_layers):
            # Passer la séquence à travers la couche layer_idx
            outputs, Hs[layer_idx] = self.rnns[layer_idx](outputs, Hs[layer_idx])
            
            # À ce stade:
            # - outputs a la forme (seq_len, batch_size, num_hiddens)
            # - Hs[layer_idx] a la forme (batch_size, num_hiddens)
        
        # Appliquer la couche de sortie finale
        # outputs est actuellement (seq_len, batch_size, num_hiddens)
        # On veut obtenir (seq_len, batch_size, num_inputs)
        seq_len, batch_size, _ = outputs.shape
        Y = []
        for t in range(seq_len):
            y_t = outputs[t] @ self.W_hq + self.b_q
            Y.append(y_t.unsqueeze(0))
        
        outputs = torch.cat(Y, dim=0)  # (seq_len, batch_size, num_inputs)
        
        return outputs, Hs


print("✓ Deep RNN défini avec succès")
print(f"  - Architecture: plusieurs couches récurrentes empilées")
print(f"  - Chaque couche l reçoit la sortie de la couche l-1")
print(f"  - Chaque couche maintient son propre état caché temporel")

## Cellule 3 : Comparaison expérimentale - RNN Simple vs Deep RNN

### Expérience
Nous allons:
1. Créer des données synthétiques (séquences de caractères)
2. Entraîner un RNN simple et un Deep RNN sur la même tâche
3. Comparer:
   - Les courbes d'apprentissage
   - Les performances de générer du texte
   - La capacité à modéliser des dépendances longues

### Code de comparaison

In [ ]:
text = "the quick brown fox jumps over the lazy dog and then it jumps again " * 10
chars = sorted(list(set(text)))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}
vocab_size = len(chars)

print(f"Vocabulaire: {chars}")
print(f"Taille du vocabulaire: {vocab_size}")
print(f"Longueur du texte: {len(text)}")

# Convertir le texte en indices
text_indices = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

# One-hot encoding
def to_onehot(indices, vocab_size):
    """Convertir des indices en vecteurs one-hot."""
    return torch.eye(vocab_size)[indices]

text_onehot = to_onehot(text_indices, vocab_size)  # (seq_len, vocab_size)

print(f"\nShape après one-hot: {text_onehot.shape}")
print(f"Exemple d'entrée (premier caractère '{text[0]}'): {text_onehot[0]}")

# 2. PARAMÈTRES D'EXPÉRIENCE
num_steps = 32       # Longueur des sous-séquences
batch_size = 32      # Nombre de séquences parallèles
num_hiddens = 16     # Dimension de l'état caché
learning_rate = 0.01
num_epochs = 50

# 3. CRÉER LES MODÈLES
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# RNN simple (1 couche)
model_simple = SimpleRNNScratch(vocab_size, num_hiddens, sigma=0.01).to(device)
optimizer_simple = Adam(model_simple.parameters(), lr=learning_rate)

# Deep RNN (3 couches)
model_deep = DeepRNNScratch(vocab_size, num_hiddens, num_layers=3, sigma=0.01).to(device)
optimizer_deep = Adam(model_deep.parameters(), lr=learning_rate)

print(f"\nModèles créés sur device: {device}")
print(f"RNN Simple: {sum(p.numel() for p in model_simple.parameters())} paramètres")
print(f"Deep RNN (3 couches): {sum(p.numel() for p in model_deep.parameters())} paramètres")

# 4. FONCTION D'ENTRAÎNEMENT
def train_epoch(model, optimizer, data, batch_size, num_steps, vocab_size, device):
    """Entraîner le modèle pendant une époque."""
    total_loss = 0
    num_batches = 0
    
    # Créer des mini-batches
    seq_len = data.shape[0]
    num_batches_total = (seq_len - num_steps - 1) // (batch_size * num_steps)
    
    for batch_idx in range(num_batches_total):
        # Sélectionner une sous-séquence
        start_idx = batch_idx * batch_size * num_steps
        if start_idx + num_steps + 1 >= seq_len:
            break
        
        # Entrée: [t:t+num_steps], Cible: [t+1:t+num_steps+1]
        X_batch = data[start_idx:start_idx + num_steps].unsqueeze(1).to(device)  # (num_steps, 1, vocab_size)
        Y_batch = data[start_idx + 1:start_idx + num_steps + 1].unsqueeze(1).to(device)  # (num_steps, 1, vocab_size)
        
        # Forward pass
        outputs, _ = model(X_batch)
        
        # Perte (cross-entropy)
        loss = nn.functional.mse_loss(outputs, Y_batch)  # MSE pour simplifier
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / max(num_batches, 1)

# 5. BOUCLE D'ENTRAÎNEMENT
print(f"\n{'='*70}")
print(f"ENTRAÎNEMENT sur {num_epochs} époque(s)")
print(f"{'='*70}\n")

loss_simple = []
loss_deep = []

text_onehot = text_onehot.to(device)

for epoch in range(num_epochs):
    # Entraîner RNN simple
    l_simple = train_epoch(model_simple, optimizer_simple, text_onehot, 1, num_steps, vocab_size, device)
    loss_simple.append(l_simple)
    
    # Entraîner Deep RNN
    l_deep = train_epoch(model_deep, optimizer_deep, text_onehot, 1, num_steps, vocab_size, device)
    loss_deep.append(l_deep)
    
    if (epoch + 1) % 10 == 0:
        print(f"Époque {epoch+1:2d} | RNN Simple Loss: {l_simple:.4f} | Deep RNN Loss: {l_deep:.4f}")

print(f"\nEntraînement terminé!\n")

# 6. VISUALISATION DES RÉSULTATS
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Courbe de perte
axes[0].plot(loss_simple, label='RNN Simple (1 couche)', linewidth=2, marker='o', markersize=4)
axes[0].plot(loss_deep, label='Deep RNN (3 couches)', linewidth=2, marker='s', markersize=4)
axes[0].set_xlabel('Époque', fontsize=12)
axes[0].set_ylabel('Perte (MSE)', fontsize=12)
axes[0].set_title('Comparaison des courbes d\'apprentissage', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Tableau comparatif
comparison_data = [
    ['Caractéristique', 'RNN Simple', 'Deep RNN (3 couches)'],
    ['Nombre de couches', '1', '3'],
    ['Profondeur verticale', 'Faible', 'Haute'],
    ['Nb de paramètres', f'{sum(p.numel() for p in model_simple.parameters())}', 
     f'{sum(p.numel() for p in model_deep.parameters())}'],
    ['Perte finale', f'{loss_simple[-1]:.4f}', f'{loss_deep[-1]:.4f}'],
    ['Hiérarchie de repr.', 'Une seule', 'Plusieurs niveaux'],
    ['Complexité calcul', 'Faible', 'Moyenne'],
    ['Dépendances longues', 'Limitées', 'Meilleures'],
]

axes[1].axis('off')
table = axes[1].table(cellText=comparison_data, cellLoc='center', loc='center',
                      colWidths=[0.35, 0.32, 0.33])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.2)

# Colorer l'en-tête
for i in range(3):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Colorer les lignes alternées
for i in range(1, len(comparison_data)):
    color = '#f0f0f0' if i % 2 == 0 else '#ffffff'
    for j in range(3):
        table[(i, j)].set_facecolor(color)

axes[1].set_title('Tableau Comparatif', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print(f"\n{'='*70}")
print(f"RÉSUMÉ DES OBSERVATIONS")
print(f"{'='*70}")
print(f"\n1. PERTE INITIALE:")
print(f"   - RNN Simple:    {loss_simple[0]:.4f}")
print(f"   - Deep RNN:      {loss_deep[0]:.4f}")
print(f"\n2. PERTE FINALE (après {num_epochs} époque(s)):")
print(f"   - RNN Simple:    {loss_simple[-1]:.4f}")
print(f"   - Deep RNN:      {loss_deep[-1]:.4f}")
print(f"\n3. AMÉLIORATION:")
print(f"   - RNN Simple:    {(loss_simple[0] - loss_simple[-1]) / loss_simple[0] * 100:.1f}%")
print(f"   - Deep RNN:      {(loss_deep[0] - loss_deep[-1]) / loss_deep[0] * 100:.1f}%")
print(f"\n4. AVANTAGES CLÉS DU DEEP RNN:")
print(f"   ✓ Représentations hiérarchiques sur plusieurs niveaux")
print(f"   ✓ Meilleure modélisation des dépendances complexes")
print(f"   ✓ Chaque couche se spécialise dans différents motifs")
print(f"\n5. INCONVÉNIENTS DU DEEP RNN:")
print(f"   ✗ Plus de paramètres à optimiser")
print(f"   ✗ Risques de gradients évanescents/explosifs amplifiés")
print(f"   ✗ Nécessite une initialisation et un clipping soigneux")
print(f"\n{'='*70}")

## Points clés à retenir

### Architecture
- **RNN Simple**: Une seule couche récurrente qui traite la séquence
- **Deep RNN**: Plusieurs couches empilées, chacune traitant la sortie de la couche précédente

### Flux d'information
```
RNN Simple (1 couche):
X_t → [RNN Layer] → H_t → Y_t
     ↓           ↓
     └─ H_{t-1} (état caché)

Deep RNN (3 couches):
X_t → [RNN L1] → H_t^(1) → [RNN L2] → H_t^(2) → [RNN L3] → H_t^(3) → Y_t
     ↓ H_{t-1}^(1)     ↓ H_{t-1}^(2)     ↓ H_{t-1}^(3)
```

### Quand utiliser Deep RNN?
- Séquences très longues ou complexes
- Dépendances à plusieurs niveaux d'abstraction
- Datasets suffisamment grands pour éviter l'overfitting

### Difficultés d'entraînement
1. **Gradients évanescents**: Le gradient diminue exponentiellement avec la profondeur
2. **Gradients explosifs**: À l'inverse, ils peuvent exploser
3. **Solutions**: Gradient clipping, initialisation prudente (Xavier/He), utiliser GRU/LSTM

### Exemple concret avec GRU/LSTM
En pratique, on utilise rarement des RNN simples empilés. On préfère les GRU ou LSTM:

```python
import torch.nn as nn

# Deep GRU (beaucoup plus robuste)
gru = nn.GRU(
    input_size=vocab_size,
    hidden_size=num_hiddens,
    num_layers=3,        # 3 couches
    dropout=0.5          # Régularisation
)
```

Le LSTM et GRU gèrent mieux les dépendances longues grâce à leurs méchanismes de mémoire cellulaire.